# Baseline Modeling: Fraud_Data

This notebook trains a baseline **Logistic Regression** classifier on the processed e-commerce fraud dataset. We use a stratified train/test split, apply **SMOTE only to the training set** to address class imbalance, and evaluate on an untouched holdout that reflects real-world fraud prevalence (~9%).

**Goals**
- Load the model-ready feature matrix from `data/processed/fraud_data_features.csv`
- Split data with stratification so both sets retain fraud cases
- Train a simple, interpretable baseline model
- Evaluate with metrics suited to imbalanced fraud detection
- Summarize what the baseline tells us before trying more complex models

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RANDOM_STATE, TEST_SIZE
from src.modeling import (
    compare_class_distributions,
    confusion_matrix_frame,
    load_fraud_feature_matrix,
    stratified_train_test_split,
    train_classifier,
)
from src.preprocessing import class_imbalance_summary

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

## 1. Load Processed Features

We use the engineered feature matrix produced by the feature pipeline: numeric features scaled, categoricals one-hot encoded, and the fraud label (`class`) separated from predictors.

In [ ]:
features, target = load_fraud_feature_matrix()

print(f"Feature matrix shape: {features.shape}")
print(f"Target shape: {target.shape}")
print(f"Fraud rate: {target.mean():.2%}")

class_imbalance_summary(target.to_frame(name="class"), target_column="class")

## 2. Stratified Train/Test Split

We hold out **20%** of transactions for evaluation (`TEST_SIZE = 0.2`, `RANDOM_STATE = 42`). Stratification keeps the fraud rate similar in both splits so the test set is representative of production traffic.

In [ ]:
split = stratified_train_test_split(
    features,
    target,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

print(f"Training rows: {len(split.x_train):,}")
print(f"Test rows:     {len(split.x_test):,}")
print(f"Features:      {len(split.feature_names)}")

train_dist = class_imbalance_summary(
    split.y_train.to_frame(name="class"), target_column="class"
)
test_dist = class_imbalance_summary(
    split.y_test.to_frame(name="class"), target_column="class"
)

display(
    pd.concat(
        [
            train_dist.assign(split="train"),
            test_dist.assign(split="test"),
        ],
        ignore_index=True,
    )
)

## 3. Handle Class Imbalance on Training Data Only

Fraud is the minority class (~9% of transactions). We apply **SMOTE** to the training set to synthesize additional fraud examples. The test set is **never** resampled — evaluation must reflect the natural class mix customers actually experience.

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)
x_train_resampled, y_train_resampled = smote.fit_resample(split.x_train, split.y_train)

x_train_resampled = pd.DataFrame(x_train_resampled, columns=split.feature_names)
y_train_resampled = pd.Series(y_train_resampled, name="class")

print(f"Training rows before SMOTE: {len(split.x_train):,}")
print(f"Training rows after SMOTE:  {len(x_train_resampled):,}")

distribution_comparison = compare_class_distributions(
    split.y_train,
    y_train_resampled,
    split.y_test,
)
display(distribution_comparison[["stage", "class", "count", "pct"]])

## 4. Train Baseline Logistic Regression

Logistic Regression is a strong first baseline: fast to train, easy to interpret, and a useful reference point before tree-based or gradient-boosted models. We train on SMOTE-balanced data without extra class weights, since resampling already rebalances the training set.

In [ ]:
baseline_model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
)

result = train_classifier(
    baseline_model,
    x_train_resampled,
    y_train_resampled,
    split.x_test,
    split.y_test,
    model_name="logistic_regression_baseline",
    store_training_data=True,
)

print(f"Model trained on {len(x_train_resampled):,} resampled rows.")
print(f"Evaluated on {len(split.x_test):,} untouched test rows.")

## 5. Evaluation Metrics

For fraud detection, **accuracy alone is misleading** — a model that always predicts "not fraud" would score ~91% accuracy while catching zero fraud. We report precision, recall, F1, ROC-AUC, AUC-PR, and the confusion matrix.

| Metric | Business meaning |
|--------|------------------|
| **Precision** | Of flagged transactions, how many are actually fraud? (controls false alarms) |
| **Recall** | Of all fraud, how much do we catch? (controls missed fraud) |
| **F1** | Balance between precision and recall |
| **ROC-AUC** | Overall ranking ability across thresholds |
| **AUC-PR** | Ranking quality focused on the rare fraud class — often more informative than ROC-AUC when fraud is uncommon |

In [ ]:
metrics = result.metrics

metrics_table = pd.DataFrame(
    {
        "metric": [
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "ROC-AUC",
            "AUC-PR",
        ],
        "value": [
            metrics.accuracy,
            metrics.precision,
            metrics.recall,
            metrics.f1,
            metrics.roc_auc,
            metrics.auc_pr,
        ],
    }
)
metrics_table["value"] = metrics_table["value"].map(lambda v: f"{v:.4f}")
display(metrics_table)

cm = confusion_matrix_frame(result.y_true, result.y_pred)
print("\nConfusion matrix (rows = actual, columns = predicted):")
display(cm)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["Legitimate (0)", "Fraud (1)"],
    yticklabels=["Legitimate (0)", "Fraud (1)"],
    ax=ax,
)
ax.set_xlabel("Predicted label")
ax.set_ylabel("Actual label")
ax.set_title("Confusion Matrix — Logistic Regression Baseline")
plt.tight_layout()
plt.show()

total_fraud = metrics.true_positives + metrics.false_negatives
total_legit = metrics.true_negatives + metrics.false_positives
print(
    f"Fraud caught: {metrics.true_positives:,} / {total_fraud:,} "
    f"({metrics.true_positives / total_fraud:.1%} recall)"
)
print(
    f"False alarms: {metrics.false_positives:,} / {total_legit:,} legitimate transactions "
    f"({metrics.false_positives / total_legit:.2%} of legit flagged)"
)

## 6. Precision-Recall Curve

The PR curve shows the trade-off between precision and recall across decision thresholds. For imbalanced fraud data, this is often more actionable than an ROC curve because it focuses on the minority class. The dashed line is the **no-skill baseline** (fraction of fraud in the test set).

In [ ]:
precision_vals, recall_vals, thresholds = precision_recall_curve(
    result.y_true,
    result.y_score,
)
baseline_prevalence = result.y_true.mean()

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(recall_vals, precision_vals, linewidth=2, label="Logistic Regression")
ax.axhline(
    baseline_prevalence,
    linestyle="--",
    color="gray",
    label=f"No-skill baseline ({baseline_prevalence:.1%} fraud rate)",
)
ax.set_xlabel("Recall (fraud caught)")
ax.set_ylabel("Precision (flags that are fraud)")
ax.set_title(
    f"Precision-Recall Curve — AUC-PR = {metrics.auc_pr:.3f}"
)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## 7. Baseline Interpretation

### What this model is good for

This baseline establishes whether a simple linear model can learn meaningful fraud signals from engineered features (purchase value, channel, browser, geography, timing, etc.). If AUC-PR is clearly above the no-skill line (~9% fraud rate), the features carry predictive value and more sophisticated models are worth exploring.

### How to read the results honestly

- **Accuracy** will look high (~90%+) because most transactions are legitimate. Treat it as context, not the primary success metric.
- **Recall** tells us how much fraud we would miss at the default 0.5 threshold. In operations, missed fraud has direct financial cost — recall is often prioritized early in model development.
- **Precision** tells us how noisy our fraud alerts would be. Low precision means analysts or automated blocks would investigate many false positives, increasing operational load.
- **AUC-PR** summarizes ranking quality for the rare fraud class. It is usually more informative than ROC-AUC when fraud is uncommon.

### Business takeaway

A baseline Logistic Regression is not the final production model — it is a **sanity check**. It answers: *"Do our processed features separate fraud from legitimate traffic at all?"* The confusion matrix quantifies the real trade-off: every extra fraud caught (true positive) may come with additional false alarms (false positives) that inconvenience customers or burden review teams.

Next steps typically include tuning the decision threshold to match business tolerance for missed fraud vs. false alarms, comparing tree-based models (Random Forest, XGBoost), and using SHAP to explain which features drive individual predictions.